In [1]:
setwd("/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/wes_ko_ukbb")
devtools::load_all("utils/modules/R/gwastools")
devtools::load_all("utils/modules/R/phasingtools/")
library(tidyverse)
library(ggforce)
library(data.table)
library(ggplot2)
library(ggrepel)
library(ggsci)
library(ggrastr)

i Loading gwastools

Loading required package: data.table

Loading required package: ggplot2

Loading required package: stringr

i Loading phasingtools

Loading required package: Hmisc

Loading required package: lattice

Loading required package: survival

Loading required package: Formula


Attaching package: 'Hmisc'


The following objects are masked from 'package:base':

    format.pval, units


! Skipping missing files: /gpfs3/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/wes_ko_ukbb/utils/modules/R/phasingtools/R/aggr_ser_by_site.R

! Adding files missing in collate: /gpfs3/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/wes_ko_ukbb/utils/modules/R/phasingtools/R/aggr_ser_by_pos.R, /gpfs3/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/wes_ko_ukbb/utils/modules/R/phasingtools/R/extract_chunk_id.R, /gpfs3/well/lindgren-ukbb/projects/ukbb-11867/flassen/projects/KO/wes_ko_ukbb/utils/modules/R/phasingtools/R/read_chunks_combined.R, and /gpfs3/well/li

In [2]:
# map from ENSEMBL to HGNC
bridge <- fread("/well/lindgren/flassen/ressources/genesets/genesets/data/biomart/220524_hgnc_ensg_enst_chr_pos.txt.gz")
ensembl_to_hgnc <- bridge$hgnc_symbol
names(ensembl_to_hgnc) <- bridge$ensembl_gene_id
ensembl_to_pos <- (bridge$start_position + bridge$end_position)/2
names(ensembl_to_pos) <- bridge$ensembl_gene_id
ensembl_to_contig <- bridge$chromosome_name
names(ensembl_to_contig) <- bridge$ensembl_gene_id

In [3]:


get_qq_df <- function(files){
    ribbon_p <- 0.95
    d <- do.call(rbind, lapply(files, function(f){
        stopifnot(file.exists(f))
        d <- fread(f)
        if (nrow(d) > 0){
            # read from conditional analysis if present
            if ("p.value_c" %in% colnames(d)) {
                d$p.value <- d$p.value_c
            }
            d$p.value.expt <- get_expected_p(d$p.value, na.rm = TRUE)
            n <- length(d$p.value)
            # create QQ
            dt <- data.table(
                ensembl_gene_id = d$MarkerID[order(d$p.value)],
                pvalue = sort(d$p.value),
                pvalue.observed = -log10(sort(d$p.value)),
                pvalue.expected = -log10(sort(d$p.value.expt)),
                clower = -log10(qbeta(p = (1 - ribbon_p) / 2, shape2 = n:1, shape1 = 1:n)),
                cupper = -log10(qbeta(p = (1 + ribbon_p) / 2, shape2 = n:1, shape1 = 1:n))
            )
            # regex the current phenotype and clean
            dt$prs <- grepl("locoprs", f)
            dt$analysis <- str_extract(basename(f), 'maf0to5e-2_.+_pLoF')
            dt$analysis <- gsub("maf0to5e-2_", "", dt$analysis)
            dt$analysis <- gsub("_pLoF", "", dt$analysis)
            dt$analysis <- gsub("_combined", "", dt$analysis)
                       
            # add hgnc and grch38 position
            dt$hgnc_symbol <- ensembl_to_hgnc[dt$ensembl_gene_id]
            dt$contig <- ensembl_to_contig[dt$ensembl_gene_id]
            dt$pos <- ensembl_to_pos[dt$ensembl_gene_id]
            
            # plot label if deviated from null
            dt$label <- NA 
            bool <- dt$pvalue.observed >= dt$clower & dt$pvalue.observed > 3
            dt$label[bool] <- dt$hgnc_symbol[bool]
            
 
        return(dt)
        } else {
            return(NULL)
        }
    }))
    return(d)
}


read_cond_files <- function(files, column = "p.value"){
    d <- do.call(rbind, lapply(files, function(f){
        stopifnot(file.exists(f))
        d <- fread(f)
        if ((nrow(d) > 0) & (column %in% colnames(d))){
                        
            # set up data
            dt <- data.table(
                ensembl_gene_id = d$MarkerID
            )
            
            # read files with cond 
            values <- d[[column]]
            dt$p.value <- values
            #print(f)
            
            # regex the current phenotype and clean
            dt$prs <- grepl("locoprs", f)
            dt$analysis <- str_extract(basename(f), 'maf0to5e-2_.+_pLoF')
            dt$analysis <- gsub("maf0to5e-2_", "", dt$analysis)
            dt$analysis <- gsub("_pLoF", "", dt$analysis)
            dt$analysis <- gsub("_combined", "", dt$analysis)
                       
            # add hgnc and grch38 position
            dt$hgnc_symbol <- ensembl_to_hgnc[dt$ensembl_gene_id]
            dt$contig <- ensembl_to_contig[dt$ensembl_gene_id]
            dt$pos <- ensembl_to_pos[dt$ensembl_gene_id]
            
            # plot label if deviated from null
            dt$label <- NA 
            bool <- dt$pvalue.observed >= dt$clower & dt$pvalue.observed > 3
            dt$label[bool] <- dt$hgnc_symbol[bool]
            
 
        return(dt)
        } else {
            return(NULL)
        }
    }))
}

In [4]:
pattern <- "pLoF_damaging_missense.txt.gz"
files <- list.files("data/saige/output/binary/step2/min_mac4", pattern = pattern, full.names = TRUE)
files<- files[!grepl("primary_care", files)]
files_no_cond <- files
d <- get_qq_df(files)
length(unique(d$analysis))
sum(d$pvalue == 0)

[1] 37

[1] 0

In [5]:
significance_T <- 0.05 / (1746 * 37) # 0.05 / 20000

In [6]:
# get hits that pass our sig thresholds
sig_table <- as.data.frame(table(d$pvalue < significance_T, d$analysis))
sig_table <- sig_table[sig_table$Var1 == TRUE, ]
sig_table <- sig_table[,c(2,3)]
colnames(sig_table) <- c("phenotype", "hits")

# summarize hits in vector
sig_phenos <- as.character(unique(sig_table$phenotype[sig_table$hits > 0]))
not_sig_phenos <- as.character(unique(sig_table$phenotype[sig_table$hits == 0]))
n_sig <- length(sig_phenos)

In [7]:
sig_phenos

[1] "CC"     "CD"     "DEM"    "DM_GD"  "DM_T1D" "DM_T2D" "INF"    "LC"    
[9] "PSOR"

In [8]:
# results without PRS
pattern <- "pLoF_damaging_missense.txt.gz"
no_prs_files <- list.files("data/saige/output/binary/step2/min_mac4", pattern = pattern, full.names = TRUE)
no_prs_files <- no_prs_files[!grepl("primary_care", no_prs_files)]
length(no_prs_files)

[1] 37

In [9]:
# get results from PRS
pattern <- "pLoF_damaging_missense_locoprs.txt.gz"
prs_files <- list.files("data/saige/output/binary/step2/min_mac4/", pattern = pattern, full.names = TRUE)
prs_files  <- prs_files [!grepl("primary_care", prs_files )]
length(prs_files)

[1] 13

In [10]:
# get results from cond common
pattern <- "pLoF_damaging_missense.*\\.txt\\.gz"
common_files <- list.files("data/saige/output/binary/step2_common_cond_SEP/min_mac4/", pattern = pattern, full.names = TRUE)
common_files  <- common_files[!grepl("primary_care", common_files)]
length(common_files)
#common_files <- common_files[grepl("(PSOR)|(NAFLD)",common_files)]
#common_files

[1] 3

In [11]:
# get results from rare analysis
pattern <- "pLoF_damaging_missense.*\\.txt\\.gz"
rare_files <- list.files("data/saige/output/binary/step2_rare_cond_v1116_retest/min_mac4/", pattern = pattern, full.names = TRUE)
rare_files  <- rare_files[!grepl("primary_care", rare_files)]

In [12]:
# load all data subset to significant phenotypes
d_no_prs <- read_cond_files(no_prs_files, column = "p.value")
d_no_prs <- d_no_prs[d_no_prs$analysis %in% sig_phenos,]
d_no_prs$method <- "pval"

d_prs <- read_cond_files(prs_files, column = "p.value")
d_prs <- d_prs[d_prs$analysis %in% sig_phenos,]
d_prs$method <- "pval_c_prs"

d_common <- read_cond_files(common_files, column = "p.value_c")
d_common <- d_common[d_common$analysis %in% sig_phenos,]
d_common$method <- "pval_c_prs_common"

d_rare <- read_cond_files(rare_files, column = "p.value_c")
d_rare <- d_rare[d_rare$analysis %in% sig_phenos,]
d_rare$method <- "pval_c_prs_rare"

In [13]:
#d_rare <- read_cond_files(rare_files, column = "p.value.NA_c")
#d_rare <- d_rare[d_rare$analysis %in% sig_phenos,]
#d_rare$method <- "pval_c_prs_rare"
#rare_files
#fread(common_files[1])

In [14]:
d <- rbind(d_no_prs, d_prs, d_common, d_rare)

In [15]:
pval_levels <- unique(d$method)
d$method <- factor(d$method, levels = pval_levels)

In [16]:

# sig genes per phenotype
sig_genes <- lapply(sig_phenos, function(pheno) {
    genes <- d_no_prs$ensembl_gene_id[d_no_prs$analysis %in% pheno & d_no_prs$p.value < significance_T]
    genes <- genes[grepl("ENSG", genes)] # remove variants 
    genes <- as.character(na.omit(genes))
    return(genes)
    })
names(sig_genes) <- sig_phenos

In [17]:
str(sig_genes)

List of 9
 $ CC    : chr [1:3] "ENSG00000132781" "ENSG00000138686" "ENSG00000007306"
 $ CD    : chr [1:5] "ENSG00000137959" "ENSG00000162654" "ENSG00000180801" "ENSG00000104368" ...
 $ DEM   : chr [1:2] "ENSG00000117480" "ENSG00000124875"
 $ DM_GD : chr [1:6] "ENSG00000159588" "ENSG00000114786" "ENSG00000243989" "ENSG00000253767" ...
 $ DM_T1D: chr [1:7] "ENSG00000203734" "ENSG00000148019" "ENSG00000172893" "ENSG00000171759" ...
 $ DM_T2D: chr "ENSG00000214706"
 $ INF   : chr [1:4] "ENSG00000168394" "ENSG00000012504" "ENSG00000168140" "ENSG00000108961"
 $ LC    : chr [1:3] "ENSG00000187889" "ENSG00000042445" "ENSG00000132530"
 $ PSOR  : chr "ENSG00000204520"


In [18]:
# subset to significant genes within each phenotype
sig_genes_keep <- unique(unlist(sig_genes))
phenotypes <- names(sig_genes)
d <- d[d$ensembl_gene_id %in% sig_genes_keep,]
lst_gene_subset <- lapply(phenotypes, function(pheno){
    genes <- sig_genes[[pheno]]
    d <- d[d$analysis %in% pheno & d$ensembl_gene_id %in% genes,]
    return(d)
})

d <- do.call(rbind, lst_gene_subset)
# from above
#d <- d[d$ensembl_gene_id %in% sig_genes_keep,]
#d$method <- factor(d$method)

In [19]:
# cast to a matrix with P-value bins as columns
casted <- dcast(d, analysis + ensembl_gene_id ~ method, value.var="p.value")
casted$hgnc_symbol <- ensembl_to_hgnc[casted$ensembl_gene_id]
casted$chromosome <- ensembl_to_contig[casted$ensembl_gene_id]
casted <- casted[,c("analysis", "chromosome","ensembl_gene_id","hgnc_symbol",pval_levels), with = FALSE]

# check PRS P-value
casted$pval_prs_sig <- is.na(casted$pval_c_prs)
casted$pval_prs_sig[!casted$pval_prs_sig] <- (casted$pval_c_prs < significance_T)[!casted$pval_prs_sig]

# check common sig P-value
casted$pval_prs_common_sig <- is.na(casted$pval_c_prs_common)
casted$pval_prs_common_sig[!casted$pval_prs_common_sig] <- (casted$pval_c_prs_common < significance_T)[!casted$pval_prs_common_sig]

# check rare sig P-value
casted$pval_prs_rare_sig <- is.na(casted$pval_c_prs_rare)
casted$pval_prs_rare_sig[!casted$pval_prs_rare_sig] <- (casted$pval_c_prs_rare < significance_T)[!casted$pval_prs_rare_sig]

# check all and remove sig from file
casted$likely_ch_effect <- casted$pval_prs_sig & casted$pval_prs_common_sig & casted$pval_prs_rare_sig
casted <- casted[,!grepl("sig",colnames(casted)), with = FALSE]


In [20]:
casted

analysis,chromosome,ensembl_gene_id,hgnc_symbol,pval,pval_c_prs,pval_c_prs_common,pval_c_prs_rare,likely_ch_effect
<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>
CC,19,ENSG00000007306,CEACAM7,4.16237e-07,2.51643e-03,NA,3.956791e-07,FALSE
CC,1,ENSG00000132781,MUTYH,1.50776e-11,4.68601e-12,NA,3.973146e-43,TRUE
CC,4,ENSG00000138686,BBS7,1.54946e-07,NA,NA,NA,TRUE
CD,8,ENSG00000104368,PLAT,4.13191e-08,NA,NA,8.696305e-07,FALSE
CD,20,ENSG00000124177,CHD6,1.87535e-13,NA,NA,6.168847e-15,TRUE
CD,1,ENSG00000137959,IFI44L,1.70550e-07,NA,NA,1.407747e-07,TRUE
CD,1,ENSG00000162654,GBP4,2.30994e-08,NA,NA,2.748568e-08,TRUE
CD,4,ENSG00000180801,ARSJ,5.31237e-09,NA,NA,1.765541e-06,FALSE
DEM,1,ENSG00000117480,FAAH,5.31476e-41,NA,NA,4.915172e-41,TRUE


### How many of the rare variants do we end up conditionion on?

In [21]:
files_with_markers <- list.files("data/saige/output/binary/step2_rare_cond_v1116_retest/min_mac4", pattern = ".markers.full", full.names = TRUE)

In [22]:
lst_with_markers <- lapply(files_with_markers, function(f){
    
    # note: in the future we also output the phenotupe with the file
    d <- fread(f)
    bname <- basename(f)
    phenotype <- stringr::str_extract(bname,"maf0to5e-2.+pLoF")
    phenotype <- gsub("(maf0to5e-2_)|(_pLoF)","",phenotype)
    d$phenotype <- phenotype
    return(d)
    
})

d_with_markers <- do.call(rbind, lst_with_markers)
d_with_markers <- d_with_markers[!grepl("primary_care",d_with_markers$phenotype),]
colnames(d_with_markers)[1] <- "varid"

In [23]:
"ENSG00000007306" %in% d_with_markers$ensembl_gene_id

[1] TRUE

In [24]:
files_with_knockouts <- list.files("data/knockouts/alt/", pattern = "pLoF_damaging_missense_all.tsv.gz", full.names = TRUE)

In [25]:
genes_to_keep <- as.character(unlist(sig_genes))
knockouts_to_keep <- c("Compound heterozygote", "Homozygote")
phenotypes_to_keep <- unique(d_with_markers$phenotype)

In [26]:
clean_python_list <- function(x) gsub('(\\[)|(\\])|(\\")',"",x)
null_omit <- function(lst) lst[!vapply(lst, is.null, logical(1))]

In [27]:
lst_with_knockouts <- lapply(files_with_knockouts, function(f){
    
    d <- fread(f)
    d <- d[d$gene_id %in% genes_to_keep,]
    d <- d[d$knockout %in% knockouts_to_keep,]
    d$varid <- clean_python_list(d$varid)
    d$genetic_phase <- clean_python_list(d$gts)

    # go over each row and expand by variant
    n <- nrow(d)
    if (n > 0){
        lst_with_variants <- lapply(1:n, function(idx){
            row <- d[idx, ]
            return(
                data.table(
                    gene = row$gene_id,
                    sample = row$s,
                    varid= unlist(strsplit(row$varid, split = ',')),
                    genotype = unlist(strsplit(row$genetic_phase, split = ',')),
                    knockout = row$knockout
                )
            )
        })
        # combine list with variants
        d <- do.call(rbind, lst_with_variants)
        return(d)
    } else {
        return(NULL)
    }
   

    
})

# remove NULLs,
lst_with_knockouts <- null_omit(lst_with_knockouts)
d_with_knockouts <- do.call(rbind, lst_with_knockouts)

In [28]:
d_with_knockouts$hgnc_symbol <- ensembl_to_hgnc[d_with_knockouts$gene]
d_with_knockouts$chromosome <- ensembl_to_contig[d_with_knockouts$gene]

In [44]:
d_with_knockouts

gene,sample,varid,genotype,knockout,hgnc_symbol,chromosome
<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>
ENSG00000172893,1468413,chr11:71441392:G:C,1|0,Compound heterozygote,DHCR7,11
ENSG00000172893,1468413,chr11:71444205:A:C,0|1,Compound heterozygote,DHCR7,11
ENSG00000172893,1626447,chr11:71435716:G:A,0|1,Compound heterozygote,DHCR7,11
ENSG00000172893,1626447,chr11:71441278:G:A,1|0,Compound heterozygote,DHCR7,11
ENSG00000172893,4484875,chr11:71435513:G:C,1|0,Compound heterozygote,DHCR7,11
ENSG00000172893,4484875,chr11:71444861:C:T,0|1,Compound heterozygote,DHCR7,11
ENSG00000012504,1315848,chr12:100532530:T:C,1|1,Homozygote,NR1H4,12
ENSG00000012504,1428479,chr12:100532530:T:C,1|1,Homozygote,NR1H4,12
ENSG00000012504,1635000,chr12:100532530:T:C,1|1,Homozygote,NR1H4,12


In [29]:
head(d_with_markers); head(d_with_knockouts)

varid,ac,keep_variant_ac,min_mac_filter,hash,keep_variant_ld,perfect_ld_marker,ensembl_gene_id,consequence_category,phenotype,keep
<chr>,<int>,<lgl>,<int>,<chr>,<lgl>,<chr>,<chr>,<chr>,<chr>,<lgl>
chr11:71435378:G:C,3,FALSE,4,7f1908d6,TRUE,NA,ENSG00000172893,damaging_missense,DM_T1D,FALSE
chr11:71435386:C:T,4,TRUE,4,f2d93db6,TRUE,NA,ENSG00000172893,damaging_missense,DM_T1D,TRUE
chr11:71435394:A:AGGCGGTAAGGCACTGC,1,FALSE,4,a3b7e849,TRUE,NA,ENSG00000172893,pLoF,DM_T1D,FALSE
chr11:71435398:G:A,2,FALSE,4,a8a0c0e0,TRUE,NA,ENSG00000172893,damaging_missense,DM_T1D,FALSE
chr11:71435407:C:T,8,TRUE,4,22f8c3f6,TRUE,NA,ENSG00000172893,damaging_missense,DM_T1D,TRUE
chr11:71435422:G:A,6,TRUE,4,ba27eb79,TRUE,NA,ENSG00000172893,damaging_missense,DM_T1D,TRUE


gene,sample,varid,genotype,knockout,hgnc_symbol,chromosome
<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>
ENSG00000172893,1468413,chr11:71441392:G:C,1|0,Compound heterozygote,DHCR7,11
ENSG00000172893,1468413,chr11:71444205:A:C,0|1,Compound heterozygote,DHCR7,11
ENSG00000172893,1626447,chr11:71435716:G:A,0|1,Compound heterozygote,DHCR7,11
ENSG00000172893,1626447,chr11:71441278:G:A,1|0,Compound heterozygote,DHCR7,11
ENSG00000172893,4484875,chr11:71435513:G:C,1|0,Compound heterozygote,DHCR7,11
ENSG00000172893,4484875,chr11:71444861:C:T,0|1,Compound heterozygote,DHCR7,11


In [32]:
combined_counts <- lapply(phenotypes_to_keep, function(pheno){
    d_phenotype <- d_with_markers[d_with_markers$phenotype %in% pheno,]
    phenotype_genes <- unique(d_phenotype$ensembl_gene_id)
    lst_gene_by_phenotype <- lapply(phenotype_genes, function(g){
        
        # subset to genes significant in current phenotype
        d_gene_by_phenotype <- d_phenotype[d_phenotype$ensembl_gene_id %in% g,]
        
        # setup dict for mapping knockout file
        dict_pheno_ac <- d_gene_by_phenotype$ac
        dict_pheno_csqs <- d_gene_by_phenotype$consequence_category
        dict_pheno_cond <- d_gene_by_phenotype$keep
        names(dict_pheno_ac) <- d_gene_by_phenotype$varid
        names(dict_pheno_csqs) <- d_gene_by_phenotype$varid
        names(dict_pheno_cond) <- d_gene_by_phenotype$varid
        
        # map to knockout file
        d_knockout <- d_with_knockouts[d_with_knockouts$gene %in% g,]
        d_knockout$ac <- dict_pheno_ac[d_knockout$varid]
        d_knockout$csqs <- dict_pheno_csqs[d_knockout$varid]
        d_knockout$cond <- dict_pheno_cond[d_knockout$varid]

        return(d_knockout)

    
    })
    
    # combine data
    d_gene_pheno_ac <- do.call(rbind, lst_gene_by_phenotype)
    d_gene_pheno_ac$phenotype <- pheno
    return(d_gene_pheno_ac)
    
})

counts <- do.call(rbind, combined_counts)
sort(unique(counts$hgnc_symbol))

[1] "ABHD14A-ACY1" "ACY1"         "ARSJ"         "BBS7"         "CCDC17"      
 [6] "CEACAM7"      "CEP78"        "CHD6"         "CXCL6"        "DHCR7"       
[11] "ECT2L"        "FAAH"         "FYB2"         "GAMT"         "GBP4"        
[16] "GPR142"       "IFI44L"       "IFRD2"        "KATNAL1"      "MICA"        
[21] "MUTYH"        "MYH2"         "NR1H4"        "PAH"          "PCDHGA8"     
[26] "PLAT"         "RANGRF"       "RETSAT"       "SYNE2"        "TAP1"        
[31] "VASN"         "XAF1"

In [33]:
outfile = "data/conditional/220914_gene_sample_variant_knockout_ac.txt"
fwrite(counts, outfile, sep = '\t')

In [125]:
# some numbers for the paper
count_by_sample <- counts[, colnames(counts) %in% c("sample","knockout","gene","hgnc_symbol","phenotype"), with = FALSE]
count_by_sample <- count_by_sample[!duplicated(count_by_sample),]
count_by_sample <- as.data.table(table(count_by_sample$gene, count_by_sample$knockout))
colnames(count_by_sample) <- c("ensembl_gene_id","knockout","count")

# combine and create nice table to keep in data
count_by_sample <- dcast(ensembl_gene_id~knockout,  value.var="count", data=count_by_sample)
count_by_sample$hgnc_symbol <- ensembl_to_hgnc[count_by_sample$ensembl_gene_id]
count_by_sample$chromosome <- paste0("chr",ensembl_to_contig[count_by_sample$ensembl_gene_id])
count_by_sample <- count_by_sample[rev(order(count_by_sample$`Compound heterozygote`)),]
head(count_by_sample)

ensembl_gene_id,Compound heterozygote,Homozygote,hgnc_symbol,chromosome
<chr>,<int>,<int>,<chr>,<chr>
ENSG00000204520,29,293,MICA,chr6
ENSG00000132781,19,13,MUTYH,chr1
ENSG00000125414,16,1,MYH2,chr17
ENSG00000171759,10,0,PAH,chr12
ENSG00000203734,7,4,ECT2L,chr6
ENSG00000257008,4,7,GPR142,chr17


In [35]:
# how many of the variants within each knockout do we condition on?
d_cond_phenos <- do.call(rbind, lapply(unique(counts$phenotype), function(pheno){
    
    d <- counts[counts$phenotype %in% pheno,]
    genes <- unique(d$gene)
    out_d <- do.call(rbind, lapply(genes, function(g){
        d_gene <- d[d$gene %in% g,]
        pct_cond_var <- round(sum(d_gene$cond / nrow(d_gene))*100, 3)
        out_df <- data.frame(
            phenotypes = pheno,
            analysis = gsub("_combined","",pheno),
            ensembl_gene_id = g, 
            cond_variants_pct = pct_cond_var
        )
        return(out_df)
    }))
    return(out_d)
    
}))


In [36]:
mrg <- merge(casted, d_cond_phenos, by = c("analysis","ensembl_gene_id"), all.x = TRUE)

In [37]:
fwrite(mrg, "derived/tables/220914_summary_conditional_analysis.txt", sep = "\t")
mrg

analysis,ensembl_gene_id,chromosome,hgnc_symbol,pval,pval_c_prs,pval_c_prs_common,pval_c_prs_rare,likely_ch_effect,phenotypes,cond_variants_pct
<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<chr>,<dbl>
CC,ENSG00000007306,19,CEACAM7,4.16237e-07,2.51643e-03,NA,3.956791e-07,FALSE,CC_combined,100.000
CC,ENSG00000132781,1,MUTYH,1.50776e-11,4.68601e-12,NA,3.973146e-43,TRUE,CC_combined,96.154
CC,ENSG00000138686,4,BBS7,1.54946e-07,NA,NA,NA,TRUE,CC_combined,100.000
CD,ENSG00000104368,8,PLAT,4.13191e-08,NA,NA,8.696305e-07,FALSE,CD_combined,100.000
CD,ENSG00000124177,20,CHD6,1.87535e-13,NA,NA,6.168847e-15,TRUE,CD_combined,100.000
CD,ENSG00000137959,1,IFI44L,1.70550e-07,NA,NA,1.407747e-07,TRUE,CD_combined,100.000
CD,ENSG00000162654,1,GBP4,2.30994e-08,NA,NA,2.748568e-08,TRUE,CD_combined,100.000
CD,ENSG00000180801,4,ARSJ,5.31237e-09,NA,NA,1.765541e-06,FALSE,CD_combined,100.000
DEM,ENSG00000117480,1,FAAH,5.31476e-41,NA,NA,4.915172e-41,TRUE,DEM_combined,83.333


In [178]:
# phenotypes
d_pheno <- fread("data/phenotypes/filtered_covar_phenotypes_binary.tsv.gz")
d_samples <- fread("data/phenotypes/UKBB_WES200k_filtered_binary_dec2021_phenotypes_with_imputed_genos.tsv.gz")
d_pheno <- d_pheno[d_pheno$eid %in% d_samples$eid,]

In [155]:
# get sample - gene - knockout status regardless of variant in the knockout
d_with_knockouts_simple <- d_with_knockouts[,c("gene","sample","knockout","hgnc_symbol","chromosome")]
d_with_knockouts_simple <- d_with_knockouts_simple[!duplicated(d_with_knockouts_simple ),]

In [194]:
# map out knockouts and phenotype combinations
indexes <- 1:nrow(mrg)
lst_phenos <- lapply(indexes, function(idx){
    
    # get phenotype subset
    current_gene_id <- mrg$ensembl_gene_id[idx]
    current_hgnc_id <- mrg$hgnc_symbol[idx]
    current_chrom <- mrg$chromosome[idx]
    current_phenotype <- mrg$phenotypes[idx]
    d_pheno_subset <- d_pheno[,c("eid",current_phenotype), with = FALSE]
    colnames(d_pheno_subset) <- c("sample","case")
    total_def <- sum(!is.na(d_pheno[[current_phenotype]]))
    total_udef <- sum(is.na(d_pheno[[current_phenotype]]))
    total_case <- sum(d_pheno[[current_phenotype]], na.rm = TRUE)
    total_ctrl <- nrow(d_pheno) - total_case - total_udef
    
    # map sample to phenotype
    d_with_knockouts_subset <- d_with_knockouts_simple[d_with_knockouts_simple$gene %in% current_gene_id,]
    d_with_knockouts_subset$phenotype <- current_phenotype
    d_combined <- merge(
        d_with_knockouts_subset,
        d_pheno_subset,
    )
    
    # tabulate and cast
    d_table <- as.data.table(table(d_combined$knockout, d_combined$case))
    d_table$label <- ifelse(d_table$V2, "case","ctrl")
    d_table_flat <- as.data.table(t(d_table))
    colnames(d_table_flat) <- gsub(" ","_",paste0(tolower(d_table$V1),'_', d_table$label))
    d_table_flat <- cbind(
        current_phenotype,
        current_chrom,
        current_gene_id, 
        current_hgnc_id,
        d_table_flat[3,],
        total_def,
        total_case,
        total_ctrl
    )
    
    return(d_table_flat)
    
})

count_by_pheno <- rbindlist(lst_phenos, fill=TRUE)
count_by_pheno[is.na(count_by_pheno)] <- 0
colnames(count_by_pheno)[3] <- "ensembl_gene_id"

In [195]:
# combine with main table of total counts and clean up
mrg_counts <- merge(count_by_pheno, count_by_sample, by = "ensembl_gene_id", all.x = TRUE)
mrg_counts$current_hgnc_id <- NULL
mrg_counts$current_chrom <- NULL

# count up total and order columns
mrg_counts$knockouts_case <- as.numeric(mrg_counts$compound_heterozygote_case) + as.numeric(mrg_counts$homozygote_case)
mrg_counts$knockouts_ctrl <- as.numeric(mrg_counts$compound_heterozygote_ctrl) + as.numeric(mrg_counts$homozygote_ctrl)
mrg_counts <- mrg_counts[,c(2,13,12,1,3,5,10,4,6,11,14,15,7,8,9)]

# clean output
colnames(mrg_counts) <- tolower(gsub(" ","_",colnames(mrg_counts)))
mrg_counts <- mrg_counts[order(mrg_counts$current_phenotype),]
mrg_counts


current_phenotype,chromosome,hgnc_symbol,ensembl_gene_id,compound_heterozygote_ctrl,compound_heterozygote_case,compound_heterozygote,homozygote_ctrl,homozygote_case,homozygote,knockouts_case,knockouts_ctrl,total_def,total_case,total_ctrl
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<int>,<dbl>,<dbl>,<int>,<int>,<int>
CC_combined,chr19,CEACAM7,ENSG00000007306,1,1,2,3,1,4,2,4,176594,3576,173018
CC_combined,chr1,MUTYH,ENSG00000132781,12,7,19,10,3,13,10,22,176594,3576,173018
CC_combined,chr4,BBS7,ENSG00000138686,1,1,2,0,0,0,1,1,176594,3576,173018
CD_combined,chr8,PLAT,ENSG00000104368,2,1,3,2,0,2,1,4,176594,1027,175567
CD_combined,chr20,CHD6,ENSG00000124177,1,0,1,1,1,2,1,2,176594,1027,175567
CD_combined,chr1,IFI44L,ENSG00000137959,0,0,0,4,1,5,1,4,176594,1027,175567
CD_combined,chr1,GBP4,ENSG00000162654,0,0,0,4,1,5,1,4,176594,1027,175567
CD_combined,chr4,ARSJ,ENSG00000180801,0,1,1,3,0,3,1,3,176594,1027,175567
DEM_combined,chr1,FAAH,ENSG00000117480,0,1,1,3,1,4,2,3,176594,2078,174516


In [172]:
mrg_counts

current_phenotype,chromosome,hgnc_symbol,ensembl_gene_id,compound_heterozygote_ctrl,compound_heterozygote_case,compound_heterozygote,homozygote_ctrl,homozygote_case,homozygote,knockouts_case,knockouts_ctrl
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<int>,<dbl>,<dbl>
CC_combined,chr19,CEACAM7,ENSG00000007306,1,1,2,3,1,4,2,4
CC_combined,chr1,MUTYH,ENSG00000132781,12,7,19,10,3,13,10,22
CC_combined,chr4,BBS7,ENSG00000138686,1,1,2,0,0,0,1,1
CD_combined,chr8,PLAT,ENSG00000104368,2,1,3,2,0,2,1,4
CD_combined,chr20,CHD6,ENSG00000124177,1,0,1,1,1,2,1,2
CD_combined,chr1,IFI44L,ENSG00000137959,0,0,0,4,1,5,1,4
CD_combined,chr1,GBP4,ENSG00000162654,0,0,0,4,1,5,1,4
CD_combined,chr4,ARSJ,ENSG00000180801,0,1,1,3,0,3,1,3
DEM_combined,chr1,FAAH,ENSG00000117480,0,1,1,3,1,4,2,3


# Get variants for common conditional analysis and MAF

In [38]:
# get results from cond common
pattern <- "*.markers$"
common_files <- list.files("data/saige/output/binary/step2_common_cond/min_mac4/", pattern = pattern, full.names = TRUE)
common_files  <- common_files[!grepl("primary_care", common_files)]

In [39]:
lst_common <- lapply(common_files, function(f){
    
    d <- fread(f, header = FALSE)
    colnames(d) <- "varid"
    bname <- basename(f)
    phenotype <- stringr::str_extract(bname,"maf0to5e-2.+pLoF")
    phenotype <- gsub("(maf0to5e-2_)|(_pLoF)","",phenotype)
    d$phenotype <- phenotype
    d$chr <- stringr::str_extract(d$varid, "chr[0-9]+")
    return(d)
    
})

#chroms_to_extract <- unique(common$chr)
#stopifnot(length(unique(common$varid)) == length(common$varid))
do.call(rbind, lst_common)

varid,phenotype,chr
<chr>,<chr>,<chr>
chr19:44908684:T:C,AD_combined,chr19
chr1:16201723:G:T,NAFLD_combined,chr1
chr6:29949108:T:C,PSOR_combined,chr6
chr6:30991224:G:A,PSOR_combined,chr6
chr6:31004812:G:T,PSOR_combined,chr6
chr6:31249331:CA:C,PSOR_combined,chr6
chr6:31271601:T:G,PSOR_combined,chr6
chr6:31271630:G:T,PSOR_combined,chr6
chr6:31351254:G:T,PSOR_combined,chr6


In [40]:
vep_file <- "data/mt/vep/worst_csq_by_gene_canonical/ukb_eur_wes_union_calls_200k_chrCHR.tsv.gz"
lst_vep <- lapply(lst_common[2], function(d){
    chrom <- unique(d$chr)
    vep_path <- gsub("chrCHR",chrom, vep_file)
    print(paste0("opening ",vep_path))
    vep <- fread(vep_path)
    vep <- vep[vep$varid %in% common$varid,]
    mrg <- merge(d, vep, all.x = TRUE)
    return(mrg)
    
})

[1] "opening data/mt/vep/worst_csq_by_gene_canonical/ukb_eur_wes_union_calls_200k_chr1.tsv.gz"


ERROR: Error in .checkTypos(e, names_x): Object 'common' not found amongst locus, alleles, rsid, info.AC, info.AF and 50 more


In [ ]:
lst_vep